In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_H.xpt"
csv_file = DATA_DIR / "FASTQX_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Keep an untouched copy of the pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 FASTQX_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "FASTQX_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9422:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 19:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,422 rows x 19 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value before validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if len(numeric_cols) != 19:

    raise ValueError(
        "Expected all 19 FASTQX_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached FASTQX_H.xpt pattern:
#
# PHDSESN   = 4,320
# PHAFSTHR  =   835
# PHAGUMHR  =   204
# PHAFSTMN  =   161
# PHACOFHR  =    25
# PHASUPHR  =     9
# PHAGUMMN  =     8
# PHACOFMN  =     3
# PHAALCHR  =     3
# PHAANTHR  =     2
# PHAALCMN  =     1
# PHASUPMN  =     1
#
# TOTAL = 5,572
#

EXPECTED_TINY_COUNTS = {
    "PHACOFHR": 25,
    "PHACOFMN": 3,
    "PHAALCHR": 3,
    "PHAALCMN": 1,
    "PHAGUMHR": 204,
    "PHAGUMMN": 8,
    "PHAANTHR": 2,
    "PHASUPHR": 9,
    "PHASUPMN": 1,
    "PHAFSTHR": 835,
    "PHAFSTMN": 161,
    "PHDSESN": 4320
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated FASTQX_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 5572:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if len(EXPECTED_TINY_COUNTS) != 12:

    raise ValueError(
        "Expected tiny artifacts in exactly "
        "12 variables."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 5,572-value tiny-zero "
    "pattern across 12 variables validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only the 12 proven variables.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 5572:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric source should contain.
#
# ONLY validated TINY_VALUE -> 0 values may change.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Always run before converting to Int64.
#
# Attached FASTQX_H contains NO genuine
# fractional variables.
#
# np.round() is being used only for validation.
# It does NOT alter any value.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in FASTQX_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 11. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# FASTQX_H has no genuine decimal fields.
#
# Therefore all 19 variables can safely become
# nullable Int64.
#
# NO .round() is used during conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 19 FASTQX_H variables "
    "converted to nullable Int64 without rounding."
)


# ============================================================
# 12. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. HELPER - RANGE / COUNT CHECK
# ============================================================

def check_range(
    column,
    expected_min,
    expected_max,
    expected_nonmissing,
    expected_missing,
    expected_zero=None
):

    s = df[column]


    nonmissing = int(
        s.notna()
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    minimum = int(
        s.dropna()
        .min()
    )


    maximum = int(
        s.dropna()
        .max()
    )


    zero_count = int(
        s.eq(0)
        .sum()
    )


    print(
        f"{column}: "
        f"n={nonmissing:,}, "
        f"missing={missing:,}, "
        f"range={minimum}-{maximum}, "
        f"zero={zero_count:,}"
    )


    if nonmissing != expected_nonmissing:

        raise ValueError(
            f"{column}: unexpected nonmissing count."
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column}: unexpected missing count."
        )


    if minimum != expected_min:

        raise ValueError(
            f"{column}: unexpected minimum."
        )


    if maximum != expected_max:

        raise ValueError(
            f"{column}: unexpected maximum."
        )


    if expected_zero is not None:

        if zero_count != expected_zero:

            raise ValueError(
                f"{column}: expected "
                f"{expected_zero:,} zeros but "
                f"found {zero_count:,}."
            )


    print(
        f"PASS: {column} range/count validated."
    )


# ============================================================
# 14. CDC CHECK - PHQ020
# ============================================================

#
# Coffee or tea with cream or sugar?
#
# 1 Yes = 129
# 2 No  = 9,053
# Missing = 240
#

check_frequency(
    "PHQ020",
    {
        1: 129,
        2: 9053
    },
    240
)


# ============================================================
# 15. CDC CHECK - PHACOFHR
# ============================================================

#
# Coffee/tea fast time - hours.
#
# CDC:
#
# 0-18 = 129
# Missing = 9,293
#
# Attached XPT:
#
# zero = 25
#

check_range(
    "PHACOFHR",
    expected_min=0,
    expected_max=18,
    expected_nonmissing=129,
    expected_missing=9293,
    expected_zero=25
)


# ============================================================
# 16. CDC CHECK - PHACOFMN
# ============================================================

#
# Coffee/tea fast time - minutes.
#
# CDC:
#
# 0-59 = 129
#
# Attached:
#
# zero = 3
#

check_range(
    "PHACOFMN",
    expected_min=0,
    expected_max=59,
    expected_nonmissing=129,
    expected_missing=9293,
    expected_zero=3
)


# ============================================================
# 17. PHQ020 ROUTING CHECK
# ============================================================

#
# PHACOFHR and PHACOFMN should be populated exactly
# when PHQ020 = 1.
#
# fillna(False) is used ONLY for temporary Boolean masks.
# It does NOT alter dataframe missingness.
#

coffee_yes = (
    df["PHQ020"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


coffee_hour_nonmissing = (
    df["PHACOFHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


coffee_minute_nonmissing = (
    df["PHACOFMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        coffee_yes
        !=
        coffee_hour_nonmissing
    )
) != 0:

    raise ValueError(
        "PHACOFHR routing does not match PHQ020=1."
    )


if int(
    np.sum(
        coffee_yes
        !=
        coffee_minute_nonmissing
    )
) != 0:

    raise ValueError(
        "PHACOFMN routing does not match PHQ020=1."
    )


print(
    "PASS: Coffee/tea fasting time routing validated."
)


# ============================================================
# 18. CDC CHECK - PHQ030
# ============================================================

check_frequency(
    "PHQ030",
    {
        1: 42,
        2: 9140
    },
    240
)


# ============================================================
# 19. CDC CHECK - PHAALCHR
# ============================================================

check_range(
    "PHAALCHR",
    expected_min=0,
    expected_max=21,
    expected_nonmissing=42,
    expected_missing=9380,
    expected_zero=3
)


# ============================================================
# 20. CDC CHECK - PHAALCMN
# ============================================================

check_range(
    "PHAALCMN",
    expected_min=0,
    expected_max=59,
    expected_nonmissing=42,
    expected_missing=9380,
    expected_zero=1
)


# ============================================================
# 21. PHQ030 ROUTING CHECK
# ============================================================

alcohol_yes = (
    df["PHQ030"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alcohol_hour_nonmissing = (
    df["PHAALCHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


alcohol_minute_nonmissing = (
    df["PHAALCMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        alcohol_yes
        !=
        alcohol_hour_nonmissing
    )
) != 0:

    raise ValueError(
        "PHAALCHR routing does not match PHQ030=1."
    )


if int(
    np.sum(
        alcohol_yes
        !=
        alcohol_minute_nonmissing
    )
) != 0:

    raise ValueError(
        "PHAALCMN routing does not match PHQ030=1."
    )


print(
    "PASS: Alcohol fasting-time routing validated."
)


# ============================================================
# 22. CDC CHECK - PHQ040
# ============================================================

check_frequency(
    "PHQ040",
    {
        1: 371,
        2: 8811
    },
    240
)


# ============================================================
# 23. CDC CHECK - PHAGUMHR
# ============================================================

check_range(
    "PHAGUMHR",
    expected_min=0,
    expected_max=18,
    expected_nonmissing=371,
    expected_missing=9051,
    expected_zero=204
)


# ============================================================
# 24. CDC CHECK - PHAGUMMN
# ============================================================

check_range(
    "PHAGUMMN",
    expected_min=0,
    expected_max=59,
    expected_nonmissing=371,
    expected_missing=9051,
    expected_zero=8
)


# ============================================================
# 25. PHQ040 ROUTING CHECK
# ============================================================

gum_yes = (
    df["PHQ040"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


gum_hour_nonmissing = (
    df["PHAGUMHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


gum_minute_nonmissing = (
    df["PHAGUMMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        gum_yes
        !=
        gum_hour_nonmissing
    )
) != 0:

    raise ValueError(
        "PHAGUMHR routing does not match PHQ040=1."
    )


if int(
    np.sum(
        gum_yes
        !=
        gum_minute_nonmissing
    )
) != 0:

    raise ValueError(
        "PHAGUMMN routing does not match PHQ040=1."
    )


print(
    "PASS: Gum/mints fasting-time routing validated."
)


# ============================================================
# 26. CDC CHECK - PHQ050
# ============================================================

check_frequency(
    "PHQ050",
    {
        1: 37,
        2: 9145
    },
    240
)


# ============================================================
# 27. CDC CHECK - PHAANTHR
# ============================================================

check_range(
    "PHAANTHR",
    expected_min=0,
    expected_max=38,
    expected_nonmissing=37,
    expected_missing=9385,
    expected_zero=2
)


# ============================================================
# 28. CDC CHECK - PHAANTMN
# ============================================================

#
# CDC range starts at 2 here.
#
# There are NO zero values in this field.
#

check_range(
    "PHAANTMN",
    expected_min=2,
    expected_max=57,
    expected_nonmissing=37,
    expected_missing=9385,
    expected_zero=0
)


# ============================================================
# 29. PHQ050 ROUTING CHECK
# ============================================================

antacid_yes = (
    df["PHQ050"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


antacid_hour_nonmissing = (
    df["PHAANTHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


antacid_minute_nonmissing = (
    df["PHAANTMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        antacid_yes
        !=
        antacid_hour_nonmissing
    )
) != 0:

    raise ValueError(
        "PHAANTHR routing does not match PHQ050=1."
    )


if int(
    np.sum(
        antacid_yes
        !=
        antacid_minute_nonmissing
    )
) != 0:

    raise ValueError(
        "PHAANTMN routing does not match PHQ050=1."
    )


print(
    "PASS: Antacid fasting-time routing validated."
)


# ============================================================
# 30. CDC CHECK - PHQ060
# ============================================================

check_frequency(
    "PHQ060",
    {
        1: 86,
        2: 9096
    },
    240
)


# ============================================================
# 31. CDC CHECK - PHASUPHR
# ============================================================

check_range(
    "PHASUPHR",
    expected_min=0,
    expected_max=13,
    expected_nonmissing=86,
    expected_missing=9336,
    expected_zero=9
)


# ============================================================
# 32. CDC CHECK - PHASUPMN
# ============================================================

check_range(
    "PHASUPMN",
    expected_min=0,
    expected_max=58,
    expected_nonmissing=86,
    expected_missing=9336,
    expected_zero=1
)


# ============================================================
# 33. PHQ060 ROUTING CHECK
# ============================================================

supplement_yes = (
    df["PHQ060"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


supplement_hour_nonmissing = (
    df["PHASUPHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


supplement_minute_nonmissing = (
    df["PHASUPMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        supplement_yes
        !=
        supplement_hour_nonmissing
    )
) != 0:

    raise ValueError(
        "PHASUPHR routing does not match PHQ060=1."
    )


if int(
    np.sum(
        supplement_yes
        !=
        supplement_minute_nonmissing
    )
) != 0:

    raise ValueError(
        "PHASUPMN routing does not match PHQ060=1."
    )


print(
    "PASS: Supplement fasting-time routing validated."
)


# ============================================================
# 34. CDC CHECK - PHAFSTHR
# ============================================================

#
# Total length of food fast in hours.
#
# CDC:
#
# 0-39 = 9,182
# Missing = 240
#
# Attached:
#
# zero = 835
#

check_range(
    "PHAFSTHR",
    expected_min=0,
    expected_max=39,
    expected_nonmissing=9182,
    expected_missing=240,
    expected_zero=835
)


# ============================================================
# 35. CDC CHECK - PHAFSTMN
# ============================================================

#
# Total length of food fast in minutes.
#
# CDC:
#
# 0-59 = 9,182
# Missing = 240
#
# Attached:
#
# zero = 161
#

check_range(
    "PHAFSTMN",
    expected_min=0,
    expected_max=59,
    expected_nonmissing=9182,
    expected_missing=240,
    expected_zero=161
)


# ============================================================
# 36. PHAFSTHR / PHAFSTMN MISSING-PATTERN CHECK
# ============================================================

fast_hour_nonmissing = (
    df["PHAFSTHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


fast_minute_nonmissing = (
    df["PHAFSTMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if not np.array_equal(
    fast_hour_nonmissing,
    fast_minute_nonmissing
):

    raise ValueError(
        "PHAFSTHR and PHAFSTMN "
        "missing patterns differ."
    )


print(
    "PASS: Total fasting hour/minute "
    "missing patterns match exactly."
)


# ============================================================
# 37. CDC CHECK - PHDSESN
# ============================================================

#
# Session in which participant was examined.
#
# CDC:
#
# 0 = Morning   4,320
# 1 = Afternoon 3,491
# 2 = Evening   1,611
#
# Missing = 0
#

check_frequency(
    "PHDSESN",
    {
        0: 4320,
        1: 3491,
        2: 1611
    },
    0
)


# ============================================================
# 38. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded file contained:
#
#     ordinary zeros = 0
#
# Therefore every zero after repair must exactly match
# one of the validated tiny-value corrections.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated FASTQX_H pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 5572:

    raise ValueError(
        "Unexpected final total zero count."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 5,572 validated corrections."
)


# ============================================================
# 39. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed source changes are:
#
#     5,572 TINY_VALUE -> 0
#
# No other numeric value may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 19 variables match the "
    "expected corrected XPT exactly."
)


# ============================================================
# 40. EXPORT CSV
# ============================================================

#
# Standard fidelity rule:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# FASTQX_H happens to contain only whole-number fields,
# but keeping this export pattern makes the workflow
# consistent with your Dietary/Lab files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 41. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 42. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 19 CSV variables round-trip "
    "to exact corrected XPT values."
)


# ============================================================
# 43. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 44. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 45. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 46. RAW CSV .0 CHECK
# ============================================================

#
# FASTQX_H contains NO genuine decimal variables.
#
# Therefore it is safe to check ALL 19 fields
# for unnecessary ".0".
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# PHAFSTHR
#     12.0 -> 12
#
# PHDSESN
#     0.0 -> 0
#
# Do NOT copy this global behavior to datasets
# containing genuine decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' formatting "
    "in any FASTQX_H field."
)


# ============================================================
# 47. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly matches "
    "the validated 5,572 corrections."
)


# ============================================================
# 48. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 49. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 50. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "FASTQX_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Laboratory - Fasting Questionnaire"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "PHDSESN zero restorations:",
    EXPECTED_TINY_COUNTS["PHDSESN"]
)


print(
    "PHAFSTHR zero restorations:",
    EXPECTED_TINY_COUNTS["PHAFSTHR"]
)


print(
    "PHAFSTMN zero restorations:",
    EXPECTED_TINY_COUNTS["PHAFSTMN"]
)


print(
    "PHAGUMHR zero restorations:",
    EXPECTED_TINY_COUNTS["PHAGUMHR"]
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 FASTQX_H XPT -> CSV
Rows: 9,422
Columns: 19
PASS: Structure = 9,422 rows x 19 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 5,572

Tiny values by variable:
PHDSESN     4320
PHAFSTHR     835
PHAGUMHR     204
PHAFSTMN     161
PHACOFHR      25
PHASUPHR       9
PHAGUMMN       8
PHACOFMN       3
PHAALCHR       3
PHAANTHR       2
PHAALCMN       1
PHASUPMN       1

PASS: Exact 5,572-value tiny-zero pattern across 12 variables validated.

--- VALIDATED ZERO RESTORATION ---
PHACOFHR: 25 restored to 0
PHACOFMN: 3 restored to 0
PHAALCHR: 3 restored to 0
PHAALCMN: 1 restored to 0
PHAGUMHR: 204 restored to 0
PHAGUMMN: 8 restored to 0
PHAANTHR: 2 restored to 0
PHASUPHR: 9 restored to 0
PHASUPMN: 1 restored to 0
PHAFSTHR: 835 restored to 0
PHAFSTMN: 161 restored to 0
PHDSESN: 4,320 restored to 0

PASS: 5,572 validated zero values restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALU